### https://www.kaggle.com/competitions/drawing-with-llms

In [1]:
from unsloth import FastLanguageModel
import kagglehub
import pandas as pd

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 04-07 20:27:12 [__init__.py:239] Automatically detected platform cuda.


In [2]:
import sys
sys.path.append('./utils')
from svg_processor import SVGSanitizer, SVGProcessor,svg_constraints
from svg_evaluator_siglip import SVGMetricEvaluator


This code could modify your python environment or operating system.

Review this code at https://www.kaggle.com/code/metric/svg-constraints/versions/1
or in your download cache at /home/vino/.cache/kagglehub/notebooks/metric/svg-constraints/output/versions/1

It is strongly recommended that you run this code within a container
such as Docker to provide a secure, isolated execution environment.
See https://www.kaggle.com/docs/packages for more information.

Do you want to proceed? (y)es/[no]:  y


In [3]:
import concurrent
import io
import logging
import re
import re2
import cairosvg
import kagglehub
import torch
from lxml import etree
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
device = "cuda" if torch.cuda.is_available() else "cpu"
print('device',device)

class Model:
    def __init__(self):

        #####-------for transformers Automodel----------------
        # ##Configure 4-bit quantization
        # bnb_config = BitsAndBytesConfig(
        #     load_in_4bit=True,
        #     bnb_4bit_quant_type='nf4',  # Normalized float 4
        #     bnb_4bit_use_double_quant=False,  # Second quantization layer
        #     bnb_4bit_compute_dtype=torch.float16  # Computation in FP16
        # )
        
        # self.model_path = "./lora/lora_16bit_merged_3b_r128_s1000_i1000_v1"
        # self.tokenizer = AutoTokenizer.from_pretrained(self.model_path)
        # self.model = AutoModelForCausalLM.from_pretrained(
        #     self.model_path,
        #     torch_dtype=torch.float16,
        #     #quantization_config=bnb_config,
        #     device_map="auto"
        # )
        #self.model.eval()
        ######-----------------------------------
        
        ###------- for Unsloth-------------------
        self.model_path = "./lora/lora_16bit_merged_3b_r128_s1000_i1000_v1"
        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name= self.model_path,
            max_seq_length=2048,
            dtype=torch.float16,
            load_in_4bit=False
        )
        self.model = self.model.to("cuda")
        FastLanguageModel.for_inference(self.model) # Enable native 2x faster inference
        #######------------------------------------
        
        # Check model dtype and device
        for name, param in self.model.named_parameters():
            print(f"{name}: {param.dtype} on {param.device}")
            break  # remove break to list all parameters

        
        
        self.default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
        self.constraints = svg_constraints.SVGConstraints()
        self.sanitizer = SVGSanitizer(self.constraints, self.default_svg)
    
    def get_response(self, description):

        instruction = """Generate SVG code to visually represent the following text description, while respecting the given constraints.
        <constraints>
        * **Allowed Elements:** `svg`, `path`, `circle`, `rect`, `ellipse`, `line`, `polyline`, `polygon`, `g`, `linearGradient`, `radialGradient`, `stop`, `defs`
        * **Allowed Attributes:** `viewBox`, `width`, `height`, `fill`, `stroke`, `stroke-width`, `d`, `cx`, `cy`, `r`, `x`, `y`, `rx`, `ry`, `x1`, `y1`, `x2`, `y2`, `points`, `transform`, `opacity`
        </constraints>
        
        <example>
        <description>"A red circle with a blue square inside"</description>
        ```svg
        <svg viewBox="0 0 256 256" width="256" height="256">
          <circle cx="50" cy="50" r="40" fill="red"/>
          <rect x="30" y="30" width="40" height="40" fill="blue"/>
        </svg>
        ```
        </example>
        
        
        Please ensure that the generated SVG code is well-formed, valid, and strictly adheres to these constraints.
        Focus on a clear and concise representation of the input description within the given limitations. 
        Always give the complete SVG code with nothing omitted. Never use an ellipsis.
        
        <description>"{}"</description>
        ```svg
        <svg viewBox="0 0 256 256" width="256" height="256">
        """
        
        alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.
    
        ### Instruction:
        Please write an SVG code for the given input.
    
        ### Input:
        {}
    
        ### Response:
        """
        
        #instruction= instruction.format(description)
        formatted_input = alpaca_prompt.format(description)
        #print(formatted_input)
        inputs = self.tokenizer([formatted_input], return_tensors="pt").to(device)
        #temperature=0.5, top_k=40, top_p=0.95, 
        outputs = self.model.generate(**inputs, max_new_tokens=1024, use_cache=True)
        return self.tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    
    def predict(self, description: str, max_new_tokens=2048) -> str:
        output_decoded = self.get_response(description)
        base_svg_code = SVGProcessor.clean_and_extract_svgs(output_decoded, self.default_svg)
        clean_svg_code = self.sanitizer.enforce_constraints(base_svg_code)
        return SVGProcessor.svg_conversion_check(description, clean_svg_code, self.default_svg)


device cuda


In [4]:
model=Model()

==((====))==  Unsloth 2025.3.18: Fast Llama patching. Transformers: 4.49.0. vLLM: 0.8.2.
   \\   /|    NVIDIA GeForce RTX 4070 Ti SUPER. Num GPUs = 1. Max memory: 15.693 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

model.embed_tokens.weight: torch.float16 on cuda:0


In [5]:
model.predict('sun rising in the east')

'<svg viewBox="0 0 256 256" width="256" height="256"><defs><linearGradient id="sunGradient" x1="0" y1="0" x2="0" y2="1"><stop offset="0%" stop-color="yellow"/><stop offset="100%" stop-color="orange"/></linearGradient></defs><circle cx="128" cy="128" r="50" fill="url(#sunGradient)" opacity="0.8"/><polyline points="128,50 100,100 128,150 160,100" fill="none" stroke="darkblue" stroke-width="2"/><line x1="0" y1="256" x2="256" y2="0" stroke="darkblue" stroke-width="2"/></svg>'

In [6]:
import pandas as pd
df=pd.read_csv('./drawing-with-llms/train.csv',header=[0])

df=pd.read_csv('./drawing-with-llms/svg_score_test.csv',header=[0])
df=df[df['score'] > 0.5]
df=df[['topic','svg_code']]
df.columns=['description','svg']


In [7]:
from tqdm import tqdm
tqdm.pandas()
df['svg'] = df['description'].progress_apply(lambda x: model.predict(x))

ERROR:root:SVG Parse Error: Opening and ending tag mismatch: defs line 2 and rect, line 40, column 18 (<string>, line 40). Returning default SVG.
100%|███████████████████████████████████████████| 76/76 [10:04<00:00,  7.96s/it]


In [8]:
#write csv file for new metric score
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
file_name = re.sub(r'[^a-zA-Z0-9]', '_', model.model_path)
df.to_csv(f'./io_files/pred_{file_name}_{timestamp}.csv', index=False)

In [9]:
#SigLip Score
evaluator = SVGMetricEvaluator()
df['sl_score'] = df.progress_apply(lambda row: evaluator.svg_metric(row['description'], row['svg']), axis=1)

Using device: cuda


100%|███████████████████████████████████████████| 76/76 [00:04<00:00, 16.42it/s]


In [10]:
df['sl_score'].mean()

0.2294024169606983